## David's Diabetes Data - A 2025 Deep Dive WIP

My boyfriend, David, has type 1 diabetes. He tracks his blood sugar levels with a standard glucometer and a Freestyle Libre 2 arm sensor, which should ideally be within the range of 5-7mmol. His finger prick testing, insulin injections at meal times and correction doses (of sugar or insulin) are recorded in the MySugr app alongside additional information (time of day, date and information tags). The following data set records approximately 4 years of data in 15,000 rows of data.

The **goal of this analysis** is to highlight the factors influencing low and high glucose events - some within this data set i.e. time of day, weekday vs weekend, activity and link it to external data e.g. daily weather, win or lose at daily Wordle puzzle.

# Read in data and examine it

**Step 1: Read in the .csv file**

In [ ]:
# Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Create dataframe called df that loads the data from the specified path
df = pd.read_csv('mySugrJuly25.csv')
# Check top rows of data
df.head(30)

,Date,Time,Tags,Blood Sugar Measurement (mmol/L),Insulin Injection Units (Pen),Basal Injection Units,Insulin Injection Units (pump),Insulin (Meal),Insulin (Correction),Temporary Basal Percentage,...,Location,Blood pressure,Body weight (kg),HbA1c (mmol/mol),Ketones,Food type,Medication,Timezone,Latitude,Longitude
0,24 Jul 2025,13:21:14,"Lunch,Before meal",4.6,NaN,NaN,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
1,24 Jul 2025,13:18:46,"Before meal,Breakfast",5.7,NaN,NaN,NaN,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
2,23 Jul 2025,20:07:44,"Before meal,Dinner",7.6,NaN,NaN,NaN,3.0,0.5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
3,23 Jul 2025,16:45:09,"Office work,Snack,After meal",4.6,NaN,NaN,NaN,1.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
4,23 Jul 2025,16:32:35,"Office work,After meal",4.5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
5,23 Jul 2025,13:17:25,"Lunch,Before meal",6.2,NaN,NaN,NaN,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
6,23 Jul 2025,10:02:16,"Before meal,Breakfast",4.8,NaN,NaN,NaN,2.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
7,23 Jul 2025,9:34:00,NaN,5.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
8,22 Jul 2025,22:31:30,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
9,22 Jul 2025,19:52:55,"Before meal,Dinner",4.6,NaN,NaN,NaN,6.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN


# **Step 2: Examine data**
- Here I am getting a sense of the amount of usable data, evaluating if the data columns have been parsed as the correct data type and seeing if any columns should be alterned or dropped due to missing data.

In [ ]:
# Check out the data structure and types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15643 entries, 0 to 15642
Data columns (total 28 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   Date                                                     15643 non-null  object 
 1   Time                                                     15643 non-null  object 
 2   Tags                                                     11446 non-null  object 
 3   Blood Sugar Measurement (mmol/L)                         12197 non-null  float64
 4   Insulin Injection Units (Pen)                            0 non-null      float64
 5   Basal Injection Units                                    1137 non-null   float64
 6   Insulin Injection Units (pump)                           0 non-null      float64
 7   Insulin (Meal)                                           4016 non-null   float64
 8   Insulin (Correction)      

In [ ]:
# Get a full view of the top 100 rows, which is the most recent data
# Check for rows that display differently from the majority e.g. row 8

from IPython.display import display
with pd.option_context('display.max_rows', 100, 'display.max_columns', 28):
    display(df.head(100))


,Date,Time,Tags,Blood Sugar Measurement (mmol/L),Insulin Injection Units (Pen),Basal Injection Units,Insulin Injection Units (pump),Insulin (Meal),Insulin (Correction),Temporary Basal Percentage,Temporary Basal Duration (Minutes),"Meal Carbohydrates (Grams, Factor 1)",Meal Descriptions,Activity Duration (Minutes),"Activity Intensity (1: Cosy, 2: Ordinary, 3: Demanding)",Activity Description,Steps,Note,Location,Blood pressure,Body weight (kg),HbA1c (mmol/mol),Ketones,Food type,Medication,Timezone,Latitude,Longitude
0,24 Jul 2025,13:21:14,"Lunch,Before meal",4.6,NaN,NaN,NaN,3.0,NaN,NaN,NaN,54.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
1,24 Jul 2025,13:18:46,"Before meal,Breakfast",5.7,NaN,NaN,NaN,2.0,NaN,NaN,NaN,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
2,23 Jul 2025,20:07:44,"Before meal,Dinner",7.6,NaN,NaN,NaN,3.0,0.5,NaN,NaN,54.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
3,23 Jul 2025,16:45:09,"Office work,Snack,After meal",4.6,NaN,NaN,NaN,1.5,NaN,NaN,NaN,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
4,23 Jul 2025,16:32:35,"Office work,After meal",4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
5,23 Jul 2025,13:17:25,"Lunch,Before meal",6.2,NaN,NaN,NaN,2.0,NaN,NaN,NaN,33.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
6,23 Jul 2025,10:02:16,"Before meal,Breakfast",4.8,NaN,NaN,NaN,2.5,NaN,NaN,NaN,34.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
7,23 Jul 2025,9:34:00,NaN,5.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
8,22 Jul 2025,22:31:30,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN
9,22 Jul 2025,19:52:55,"Before meal,Dinner",4.6,NaN,NaN,NaN,6.5,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GMT+01:00,NaN,NaN


In [ ]:
# Check for valid data in the Ketones column (this is an important data point as dangerous ketones are produced when blood sugar levels are too high (hyperglycemia))
# Created a new subset of data in a dataframe using the .loc method to find rows with non-null ketone data (.notnull method)
ketones_df = df.loc[df['Ketones'].notnull()]
print(ketones_df)

              Date      Time  \
187     2 Jul 2025  12:13:22   
422     7 Jun 2025  11:27:27   
544    26 May 2025  11:13:11   
757     8 May 2025  11:19:57   
760     8 May 2025   9:16:34   
829     1 May 2025  11:51:16   
1332   20 Mar 2025  12:29:32   
1434   12 Mar 2025  17:24:00   
1522    2 Mar 2025  20:22:39   
1808    2 Feb 2025  10:19:57   
1816    1 Feb 2025  11:59:25   
1843   29 Jan 2025  16:54:15   
1951   16 Jan 2025  15:39:26   
2407   29 Nov 2024  16:10:57   
2413   29 Nov 2024  10:56:05   
2707    7 Nov 2024   9:19:23   
2768   31 Oct 2024  20:16:03   
2823   26 Oct 2024  13:52:06   
2837   24 Oct 2024  22:42:46   
2839   24 Oct 2024  21:48:46   
2873   20 Oct 2024  11:34:52   
3061    2 Oct 2024  10:30:56   
3083    1 Oct 2024   9:11:42   
3600    1 Aug 2024  12:57:35   
4269   26 May 2024   8:40:42   
4676    3 May 2024   9:13:00   
4751   29 Apr 2024   9:06:16   
5010   13 Apr 2024   9:30:33   
5186    3 Apr 2024   9:27:13   
5206    2 Apr 2024  10:35:09   
5587    

# Initial observations for Data Cleaning:
# Remove rows
1. Basal Injection Units: night time Lantus (long-acting insulin) injections display in a different format to meal and blood test data, so should be isolated for separate analysis (e.g. row 8, Basal Injection Units = 10)
2. Ketones: these are entries following ketone tests and should also be isolated for separate analysis

# Remove columns
3. Null Data Columns: Multiple columns have 0 non-null values (no data) - these should be removed
4. Low Data Entry Columns: a number columns have very few entries so we should also exclude these for simplicity i.e. Meal Descriptions, Activity Duration (Minutes), Activity Description

# Change date format
5. Date and Time: these are being read as objects - need to parse as date/time
6. Order: Data is ordered by date descending (i.e. latest data first)- it may be useful to reorder it

# Re-organise tag information
7. Tags: these are in a comma-separated string which is unusable - need to restructure into multiple columns or tag their presence in columns with 1 vs 0


## 1. Basal Injection Units: Remove rows (not null)
Night time Lantus (long-acting insulin) injections display in a different format to meal and blood test data, so should be isolated for separate analysis (e.g. row 8, Basal Injection Units = 10)


In [ ]:
# Column 'Basal Injection Units' is not null when a row refers to a lantus injection - remove these
df = df[df['Basal Injection Units'].isnull()]
df.describe() # Check there are no rows with data for 'Basal Injection Units'

,Blood Sugar Measurement (mmol/L),Insulin Injection Units (Pen),Basal Injection Units,Insulin Injection Units (pump),Insulin (Meal),Insulin (Correction),Temporary Basal Percentage,Temporary Basal Duration (Minutes),"Meal Carbohydrates (Grams, Factor 1)",Activity Duration (Minutes),...,Steps,Location,Blood pressure,Body weight (kg),HbA1c (mmol/mol),Ketones,Food type,Medication,Latitude,Longitude
count,11987.000000,0.0,0.0,0.0,4013.000000,769.000000,0.0,0.0,5828.000000,13.000000,...,0.0,0.0,0.0,0.0,0.0,55.000000,0.0,0.0,0.0,0.0
mean,6.242204,NaN,NaN,NaN,4.075903,0.982445,NaN,NaN,35.339739,58.846154,...,NaN,NaN,NaN,NaN,NaN,0.149091,NaN,NaN,NaN,NaN
std,2.062566,NaN,NaN,NaN,2.508929,0.799180,NaN,NaN,22.482048,74.976492,...,NaN,NaN,NaN,NaN,NaN,0.176250,NaN,NaN,NaN,NaN
min,1.700000,NaN,NaN,NaN,0.500000,0.500000,NaN,NaN,2.000000,15.000000,...,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
25%,4.700000,NaN,NaN,NaN,3.000000,0.500000,NaN,NaN,12.000000,20.000000,...,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
50%,5.900000,NaN,NaN,NaN,4.000000,1.000000,NaN,NaN,34.000000,40.000000,...,NaN,NaN,NaN,NaN,NaN,0.100000,NaN,NaN,NaN,NaN
75%,7.500000,NaN,NaN,NaN,5.500000,1.000000,NaN,NaN,50.000000,60.000000,...,NaN,NaN,NaN,NaN,NaN,0.200000,NaN,NaN,NaN,NaN
max,21.000000,NaN,NaN,NaN,72.000000,12.000000,NaN,NaN,166.000000,300.000000,...,NaN,NaN,NaN,NaN,NaN,0.600000,NaN,NaN,NaN,NaN


## 2. Ketones: Remove rows (not null)
 These are row entries following ketone tests and should also be isolated for separate analysis

In [ ]:
# Column 'Ketones' holds data for ketone tests and should also be removed
df = df[df['Ketones'].isnull()]
df.describe() # Check there are no rows with data for 'Ketones'

,Blood Sugar Measurement (mmol/L),Insulin Injection Units (Pen),Basal Injection Units,Insulin Injection Units (pump),Insulin (Meal),Insulin (Correction),Temporary Basal Percentage,Temporary Basal Duration (Minutes),"Meal Carbohydrates (Grams, Factor 1)",Activity Duration (Minutes),...,Steps,Location,Blood pressure,Body weight (kg),HbA1c (mmol/mol),Ketones,Food type,Medication,Latitude,Longitude
count,11951.000000,0.0,0.0,0.0,4006.000000,754.000000,0.0,0.0,5823.000000,13.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,6.224500,NaN,NaN,NaN,4.075911,0.966844,NaN,NaN,35.346213,58.846154,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,2.033436,NaN,NaN,NaN,2.509046,0.788927,NaN,NaN,22.489556,74.976492,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.700000,NaN,NaN,NaN,0.500000,0.500000,NaN,NaN,2.000000,15.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,4.700000,NaN,NaN,NaN,3.000000,0.500000,NaN,NaN,12.000000,20.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,5.900000,NaN,NaN,NaN,4.000000,1.000000,NaN,NaN,34.000000,40.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,7.400000,NaN,NaN,NaN,5.500000,1.000000,NaN,NaN,50.000000,60.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,21.000000,NaN,NaN,NaN,72.000000,12.000000,NaN,NaN,166.000000,300.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. and 4. Remove null and low data columns

In [ ]:
# Multiple columns have 0 non-null values (no data) - these should be removed
# Many columns have very low amounts of data (e.g. 12 rows) - these should also be removed
# New, useful data set with reduced columns
df = df[['Date',
             'Time',
             'Tags',
             'Blood Sugar Measurement (mmol/L)',
             'Insulin (Meal)',
             'Insulin (Correction)',
             'Meal Carbohydrates (Grams, Factor 1)']]

df.head()

,Date,Time,Tags,Blood Sugar Measurement (mmol/L),Insulin (Meal),Insulin (Correction),"Meal Carbohydrates (Grams, Factor 1)"
0,24 Jul 2025,13:21:14,"Lunch,Before meal",4.6,3.0,NaN,54.0
1,24 Jul 2025,13:18:46,"Before meal,Breakfast",5.7,2.0,NaN,22.0
2,23 Jul 2025,20:07:44,"Before meal,Dinner",7.6,3.0,0.5,54.0
3,23 Jul 2025,16:45:09,"Office work,Snack,After meal",4.6,1.5,NaN,28.0
4,23 Jul 2025,16:32:35,"Office work,After meal",4.5,NaN,NaN,NaN


## 5. Clean Date and Time columns to read as date/time variables

In [ ]:
# 1. Date and Time: these are being read as objects - we need to parse these columns as date/time

# Problem 1: 'Date' is currently in the format '%d %b %Y' (e.g. 10 Jun 2025)
# Problem 2: "30 Sept 2024" fails because "Sept" is not a standard abbreviation for September that pandas recognizes with %b.
# The standard abbreviation is typically "Sep", so we must find the instances of 'Sept' in the 'Date' column and replace them with 'Sep'

# Solution: From 'Date' (currently a string), extract 'Day' (first 1-2 characters), 'Year' (last 4 characters) and Month (middle 3-4 characters

# Year - this is the easiest. Take the last 4 characters of the string 'Date'
df['Year'] = df['Date'].str[-4:]
df['Year'].value_counts() # confirms the dates 2021-2025 have been successfully extracted


,count
Year,
2023,4744
2024,4226
2021,2365
2025,1927
2022,1189


In [ ]:
# Day - using the .str.partition() method, the day is partitioned from the month and year (which have remained grouped as a string because of the inconsistent formatting)
day = df['Date'].str.partition(' ')[0]

# Check values lie between 1 and 31 - they do! (Note: the data is currently an array)
day.unique()

# Convert the array to a series so it can be appended to the original dataframe
df['Day'] = pd.Series(day)
df.head(10)


,Date,Time,Tags,Blood Sugar Measurement (mmol/L),Insulin (Meal),Insulin (Correction),"Meal Carbohydrates (Grams, Factor 1)",Year,Day
0,24 Jul 2025,13:21:14,"Lunch,Before meal",4.6,3.0,NaN,54.0,2025,24
1,24 Jul 2025,13:18:46,"Before meal,Breakfast",5.7,2.0,NaN,22.0,2025,24
2,23 Jul 2025,20:07:44,"Before meal,Dinner",7.6,3.0,0.5,54.0,2025,23
3,23 Jul 2025,16:45:09,"Office work,Snack,After meal",4.6,1.5,NaN,28.0,2025,23
4,23 Jul 2025,16:32:35,"Office work,After meal",4.5,NaN,NaN,NaN,2025,23
5,23 Jul 2025,13:17:25,"Lunch,Before meal",6.2,2.0,NaN,33.0,2025,23
6,23 Jul 2025,10:02:16,"Before meal,Breakfast",4.8,2.5,NaN,34.0,2025,23
7,23 Jul 2025,9:34:00,NaN,5.4,NaN,NaN,NaN,2025,23
9,22 Jul 2025,19:52:55,"Before meal,Dinner",4.6,6.5,NaN,113.0,2025,22
10,22 Jul 2025,13:24:29,"Lunch,Before meal",4.5,1.5,NaN,33.0,2025,22


In [ ]:
# Month - this can be extracted from the second part of the partition, referencing the text that comes before the space (which we know is at index -5)
month_year = df['Date'].str.partition(' ')[2]
month = month_year.str[:-5]
# Check for just the 12 months
month.unique()


array(['Jul', 'Jun', 'May', 'Apr', 'Mar', 'Feb', 'Jan', 'Dec', 'Nov',
       'Oct', 'Sept', 'Aug'], dtype=object)

In [ ]:
# Change the instances of 'Sept' to 'Sep'
# Convert the array to a series so it can be appended to the original dataframe
df['Month'] = pd.Series(month)
df.loc[df.Month == 'Sept', 'Month'] = "Sep"

# Check if all Septembers are 'Sep'
df['Month'].unique()

array(['Jul', 'Jun', 'May', 'Apr', 'Mar', 'Feb', 'Jan', 'Dec', 'Nov',
       'Oct', 'Sep', 'Aug'], dtype=object)

In [ ]:
# Create new 'Date_New' variable from our new 'Day', 'Month' and 'Year' columns by concatenating these string variables
df['Date_Str'] = df['Day'] + ' ' + df['Month'] + ' ' + df['Year']
df.head()

# Convert to the standard pandas format of '%Y-%m-%d' using .to_datetime()
df['Date_New'] = pd.to_datetime(df['Date_Str'], format='%d %b %Y')
# df['date'] = pd.to_datetime(dict(year=df['Year'], month=df['Month_Number'], day=df['Day']))

df.sample(20)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14451 entries, 0 to 15642
Data columns (total 12 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Date                                  14451 non-null  object        
 1   Time                                  14451 non-null  object        
 2   Tags                                  11233 non-null  object        
 3   Blood Sugar Measurement (mmol/L)      11951 non-null  float64       
 4   Insulin (Meal)                        4006 non-null   float64       
 5   Insulin (Correction)                  754 non-null    float64       
 6   Meal Carbohydrates (Grams, Factor 1)  5823 non-null   float64       
 7   Year                                  14451 non-null  object        
 8   Day                                   14451 non-null  object        
 9   Month                                 14451 non-null  object        
 10  Dat

In [ ]:
# Convert 'Time' to a date/time variable, using just the time format
df['Time_New'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.strftime('%H:%M:%S')

df['Date_Time'] = pd.to_datetime(df['Date_New'].astype(str) + ' ' + df['Time_New'].astype(str))
df.info()
df.head(10)

<class 'pandas.core.frame.DataFrame'>
Index: 14451 entries, 0 to 15642
Data columns (total 14 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Date                                  14451 non-null  object        
 1   Time                                  14451 non-null  object        
 2   Tags                                  11233 non-null  object        
 3   Blood Sugar Measurement (mmol/L)      11951 non-null  float64       
 4   Insulin (Meal)                        4006 non-null   float64       
 5   Insulin (Correction)                  754 non-null    float64       
 6   Meal Carbohydrates (Grams, Factor 1)  5823 non-null   float64       
 7   Year                                  14451 non-null  object        
 8   Day                                   14451 non-null  object        
 9   Month                                 14451 non-null  object        
 10  Dat

,Date,Time,Tags,Blood Sugar Measurement (mmol/L),Insulin (Meal),Insulin (Correction),"Meal Carbohydrates (Grams, Factor 1)",Year,Day,Month,Date_Str,Date_New,Time_New,Date_Time
0,24 Jul 2025,13:21:14,"Lunch,Before meal",4.6,3.0,NaN,54.0,2025,24,Jul,24 Jul 2025,2025-07-24,13:21:14,2025-07-24 13:21:14
1,24 Jul 2025,13:18:46,"Before meal,Breakfast",5.7,2.0,NaN,22.0,2025,24,Jul,24 Jul 2025,2025-07-24,13:18:46,2025-07-24 13:18:46
2,23 Jul 2025,20:07:44,"Before meal,Dinner",7.6,3.0,0.5,54.0,2025,23,Jul,23 Jul 2025,2025-07-23,20:07:44,2025-07-23 20:07:44
3,23 Jul 2025,16:45:09,"Office work,Snack,After meal",4.6,1.5,NaN,28.0,2025,23,Jul,23 Jul 2025,2025-07-23,16:45:09,2025-07-23 16:45:09
4,23 Jul 2025,16:32:35,"Office work,After meal",4.5,NaN,NaN,NaN,2025,23,Jul,23 Jul 2025,2025-07-23,16:32:35,2025-07-23 16:32:35
5,23 Jul 2025,13:17:25,"Lunch,Before meal",6.2,2.0,NaN,33.0,2025,23,Jul,23 Jul 2025,2025-07-23,13:17:25,2025-07-23 13:17:25
6,23 Jul 2025,10:02:16,"Before meal,Breakfast",4.8,2.5,NaN,34.0,2025,23,Jul,23 Jul 2025,2025-07-23,10:02:16,2025-07-23 10:02:16
7,23 Jul 2025,9:34:00,NaN,5.4,NaN,NaN,NaN,2025,23,Jul,23 Jul 2025,2025-07-23,09:34:00,2025-07-23 09:34:00
9,22 Jul 2025,19:52:55,"Before meal,Dinner",4.6,6.5,NaN,113.0,2025,22,Jul,22 Jul 2025,2025-07-22,19:52:55,2025-07-22 19:52:55
10,22 Jul 2025,13:24:29,"Lunch,Before meal",4.5,1.5,NaN,33.0,2025,22,Jul,22 Jul 2025,2025-07-22,13:24:29,2025-07-22 13:24:29


In [ ]:
df_final = df[['Date_New',
              'Date_Time',
              'Month',
              'Year',
              'Blood Sugar Measurement (mmol/L)',
              'Insulin (Meal)',
              'Insulin (Correction)',
              'Meal Carbohydrates (Grams, Factor 1)',
              'Tags']]

df_final.head()

,Date_New,Date_Time,Month,Year,Blood Sugar Measurement (mmol/L),Insulin (Meal),Insulin (Correction),"Meal Carbohydrates (Grams, Factor 1)",Tags
0,2025-07-24,2025-07-24 13:21:14,Jul,2025,4.6,3.0,NaN,54.0,"Lunch,Before meal"
1,2025-07-24,2025-07-24 13:18:46,Jul,2025,5.7,2.0,NaN,22.0,"Before meal,Breakfast"
2,2025-07-23,2025-07-23 20:07:44,Jul,2025,7.6,3.0,0.5,54.0,"Before meal,Dinner"
3,2025-07-23,2025-07-23 16:45:09,Jul,2025,4.6,1.5,NaN,28.0,"Office work,Snack,After meal"
4,2025-07-23,2025-07-23 16:32:35,Jul,2025,4.5,NaN,NaN,NaN,"Office work,After meal"


In [ ]:
df_final = df_final.rename(columns={'Date_New': 'Date',
                              'Blood Sugar Measurement (mmol/L)': 'Blood Sugar',
                              'Meal Carbohydrates (Grams, Factor 1)': 'Carbohydrates (g)'})
df_final.info()


<class 'pandas.core.frame.DataFrame'>
Index: 14451 entries, 0 to 15642
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  14451 non-null  datetime64[ns]
 1   Date_Time             14451 non-null  datetime64[ns]
 2   Month                 14451 non-null  object        
 3   Year                  14451 non-null  object        
 4   Blood Sugar           11951 non-null  float64       
 5   Insulin (Meal)        4006 non-null   float64       
 6   Insulin (Correction)  754 non-null    float64       
 7   Carbohydrates (g)     5823 non-null   float64       
 8   Tags                  11233 non-null  object        
dtypes: datetime64[ns](2), float64(4), object(3)
memory usage: 1.6+ MB


# Re-organise tag information
7. Tags: these are in a comma-separated string which is unusable - need to restructure into multiple columns or tag their presence in columns with 1 vs 0

Unfortunatly, the combinations of tags are considered as unique strings. I have to find a way to separate them

In [ ]:
# Find the unique instances of 'Tags'
df['Tags'].unique()

array(['Lunch,Before meal', 'Before meal,Breakfast', 'Before meal,Dinner',
       'Office work,Snack,After meal', 'Office work,After meal', nan,
       'Stress,Office work,After meal',
       'Hyperglycemia feeling,After meal', 'After meal',
       'Snack,After meal', 'Bedtime', 'Hypoglycemia feeling,After meal',
       'Hypoglycemia feeling', 'Lunch,Before meal,Carb guess',
       'Lunch,Eating out,Before meal,Carb guess', 'After meal,Bedtime',
       'Before meal,Carb guess,Dinner', 'Foot on the floor',
       'Hypoglycemia feeling,Lunch,Eating out,Tired,Before meal,Carb guess',
       'Hypoglycemia feeling,After meal,Dinner',
       'Hypoglycemia feeling,Snack,Before meal,After meal',
       'Driving,Hypoglycemia feeling,On the way,After meal',
       'Office work,Tired,After meal',
       'Eating out,Lunch,Before meal,Carb guess',
       'Hypoglycemia feeling,Office work,After meal',
       'Hypoglycemia feeling,Before meal,Dinner',
       'Lunch,Tired,Before meal', 'Office work,Sn

In [ ]:
# Create new column 'Commas' and count the number of commas that appear in the column 'Tags'
df["Commas"] = df.Tags.str.count(',')
df['Commas'].head()

,Commas
0,1.0
1,1.0
2,1.0
3,2.0
4,1.0


Option 2: Try a pandas solution (yay, it worked!)

In [ ]:
tags = df.Tags.str.split(',',expand=True).stack().unique().tolist()
print(tags)

['Lunch', 'Before meal', 'Breakfast', 'Dinner', 'Office work', 'Snack', 'After meal', 'Stress', 'Hyperglycemia feeling', 'Bedtime', 'Hypoglycemia feeling', 'Carb guess', 'Eating out', 'Foot on the floor', 'Tired', 'Driving', 'On the way', 'Manual work', 'Headache', 'Special situation', 'At night', 'Travelling', 'Vacation', 'Housework', 'Nervous', 'Excited', 'Angry', 'Sad', 'After sports', 'Sick', 'Pain', 'Party', 'Happy', 'Fasting', 'Chilling', 'Shopping', 'Sports', 'Correction', 'Allergies']


Create binary variable to indicate the presence of each tag in a column

In [ ]:
# Test out by first creating a variable 'Lunch' WRONG
if "Lunch" in df.Tags == True:
  df['Lunch'] = 1
else:
  df['Lunch'] = 0

df.head()

In [ ]:
df['Lunch1'] = (df.Tags.str.contains('Lunch').map({True : '1', False : '0'}))
df.head()